## Apresentação 

Notebook destinado a conter aspectos básicos de engenharia de prompt (prompt engineering), estando relacionado às minhas anotações provenientes do livro [Prompt Engineering for Generative AI](https://www.oreilly.com/library/view/prompt-engineering-for/9781098153427/). O mote do livro se dá na demonstração e explicação dos prompts utilizados os quais são elaborados com o intento de melhorar a qualidade de resposta do modelo generativo. O livro aborda, em especial, prompts para modelos textuais, mas também visuais, voltados aos modelos de difusão, como o Midjourney por exemplo. 

No presente notebook trago a respeito de formatação das saídas geradas pelos modelos de linguagem generativa (LLM), a saber :

- saída simples

- saída utilizando formato JSON

- saída utilizando formato Yaml

- saída utilizando parsers por meio do Pydantic

Cada tipo de saída melhor se direciona ao contexto da aplicação generativa da qual dispõe, sendo os três últimos uma forma de estruturar a mensagem gerada pela LLM segundo formatos que podem prover melhor manipulação do conteúdo gerado no cenário produtivo. Além disso, há uma parte em que abordo a possibilidade de extração de `features` ou características, visando garantir a sintentização de textos que compreendam as características do primeiro, como tom de voz, estilo, tema e afins. 

Tal possibilidade pode ofercer particular vantagem em cenários em que precisa-se sintetizar dados para servir de avaliação da qualidade de resposta do modelo generativo - como mensagens de usuários - ou de ensinar ao modelo a como analisar a resposta informada em função de um texto e de suas características elaboradas por um curador, circunscrevendo-se no contexto de LLM as a judge. Tais aspectos são abordados tanto no presente livro quanto no livro [AI Engineering](https://www.oreilly.com/library/view/ai-engineering/9781098166298/), da Chip Huyen. 

### Library 


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [58]:
import getpass
import os
from pprint import pprint
from typing import List

from IPython.display import Markdown
from langchain.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field

# import pandas

### Inicializando o modelo 

In [3]:
# api_key_example = "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"

os.environ["GROQ_API_KEY"]=getpass.getpass("Your api key: ")

In [4]:
llm = ChatGroq(
    model       = "llama3-70b-8192", 
    temperature = 0.2
)

In [79]:
# Testando a conexão com o modelo

response = llm.invoke("Defina em uma frase a música There is a light that never goes out do The Smiths").content
pprint(response)

('"There Is a Light That Never Goes Out" é uma canção do The Smiths que '
 'captura a intensidade e a dramaticidade de um amor obsessivo e possessivo, '
 'com letras que oscilam entre a paixão e a morte, criando uma atmosfera '
 'sombria e ao mesmo tempo bela.')


### Verificando o formato de saída para cada resposta 

Os formatos que se apresentam se mostram na forma simples, sem a especificação de nenhuma estrutura com a qual a resposta precisa ser estruturada, sendo seguida por uma estrutura JSON e Yaml. A principal vantagem das saídas estruturadas é que elas podem ser melhor trabalhadas num contexto de desenvolvimento de aplicações no cenário produivo, permitindo por exemplo extrair apenas um trecho da resposta delimitada por chaves. 

Um ponto de atenção, no entanto, é que sempre é recomendado que para modelos menos robustos seja especificado um exemplo que sirva de instrunção para o modelo compreender como deve ser realizada a estrutura esperada. Ainda, o formato Yaml em relação ao JSON oferece particular vantagem em termos de visualização e escrita, admitindo comentários realizados, como forma de facilitar a interpretação a um futuro leitor. 

In [9]:
sys_simple_message = """\
    Aja como uma secretária de consultório de uma clínica psicóligica. 
    A sua tarefa é realizar o processo de conversa com o cliente para a formação da agenda do psicólogo do consultório. 
    Apresente uma fala empática, clara e acolhedora, buscando trazer conforto às pessoas que procurarem o consultório. 
    Seja prestativa e atenciosa, buscando promover a sensação de acolhimento. 
    Não seja demasiadamente prolixa em sua fala, sendo clara, informativa e assertiva. 
    Sempre se apresente com saudações e finalize o atendimento com um agradecimento. 
"""

sys_json_message = """\
    Aja como uma secretária de consultório de uma clínica psicóligica. 
    A sua tarefa é realizar o processo de conversa com o cliente para a formação da agenda do psicólogo do consultório. 
    Apresente uma fala empática, clara e acolhedora, buscando trazer conforto às pessoas que procurarem o consultório. 
    Seja prestativa e atenciosa, buscando promover a sensação de acolhimento. 
    Não seja demasiadamente prolixa em sua fala, sendo clara, informativa e assertiva. 
    Sempre se apresente com saudações e finalize o atendimento com um agradecimento. 

    Forneça a sua resposta sempre segundo uma estrutura JSON. 
"""

sys_yaml_message = """\
    Aja como uma secretária de consultório de uma clínica psicóligica. 
    A sua tarefa é realizar o processo de conversa com o cliente para a formação da agenda do psicólogo do consultório. 
    Apresente uma fala empática, clara e acolhedora, buscando trazer conforto às pessoas que procurarem o consultório. 
    Seja prestativa e atenciosa, buscando promover a sensação de acolhimento. 
    Não seja demasiadamente prolixa em sua fala, sendo clara, informativa e assertiva. 
    Sempre se apresente com saudações e finalize o atendimento com um agradecimento. 

    Forneça a sua resposta sempre segundo uma estrutura de Yaml. 
"""

In [10]:
# Normal output :

normal_output = llm.invoke(sys_simple_message).content
print(normal_output)

Olá! Bem-vindo ao nosso consultório de psicologia. Meu nome é [nome], e sou a secretária responsável por agendar as consultas com nossos psicólogos. Estou aqui para ajudá-lo a encontrar um horário que atenda às suas necessidades.

Como você está se sentindo hoje? O que o trouxe até nosso consultório?


In [ ]:
# JSON output : 

json_output = llm.invoke(sys_json_message).content
print(json_output)

Olá! Bem-vindo(a) ao nosso consultório de psicologia. Meu nome é [nome], e sou a secretária responsável por agendar os atendimentos com nossos psicólogos. Estou aqui para ajudá-lo(a) a encontrar um horário que atenda às suas necessidades.

**Resposta JSON**
```json
{
  "saudacao": "Olá! Bem-vindo(a) ao nosso consultório de psicologia.",
  "apresentacao": "Meu nome é [nome], e sou a secretária responsável por agendar os atendimentos com nossos psicólogos.",
  "objetivo": "Estou aqui para ajudá-lo(a) a encontrar um horário que atenda às suas necessidades.",
  "pergunta": "Posso saber seu nome e qual é o motivo que o trouxe ao nosso consultório?"
}
```

Aguardo sua resposta!


In [ ]:
# Yaml output : 

yaml_output = llm.invoke(sys_yaml_message).content
print(yaml_output)

**Início do Atendimento**

nome: Secretária do Consultório
mensagem: Olá! Seja bem-vindo(a) ao nosso consultório psicológico. Meu nome é [nome], e estou aqui para ajudá-lo(a) a agendar uma consulta com um de nossos psicólogos. Como você está se sentindo hoje?

**Aguardando Resposta do Cliente**

(Pausa para ouvir a resposta do cliente)

**Resposta do Cliente**

(Insira a resposta do cliente aqui)

**Resposta da Secretária**

nome: Secretária do Consultório
mensagem: Entendi. Estamos aqui para ajudá-lo(a) a trabalhar em seus objetivos e melhorar sua saúde mental. Posso saber um pouco mais sobre o que o trouxe até aqui hoje? Qual é o seu principal motivo para buscar ajuda psicológica?

**Aguardando Resposta do Cliente**

(Pausa para ouvir a resposta do cliente)

**Resposta do Cliente**

(Insira a resposta do cliente aqui)

**Resposta da Secretária**

nome: Secretária do Consultório
mensagem: Okay, entendi. Estamos aqui para apoiá-lo(a). Vamos agendar uma consulta com um de nossos psicólo

### Estruturando as saídas com Pydantic

O Pydantic é uma biblioteca Python de código aberto que oferece uma maneira simples e elegante de validar dados, sendo amplamente utilizada na comunidade Python para validar dados em aplicativos web, APIs, análise de dados etc. Por meio dela, é possível especificar um formato de saída esperado para o modelo de LLM, sendo esse normalmente chamado de `parser`. 
 

In [59]:
class BandName(BaseModel):
    name: str = Field(description="O nome da banda de música")
    rating_score: float = Field(description="""\
                                A pontuação para o nome da banda. O é o pior e 10 o melhor. 
                                """)
    
class BandsNames(BaseModel):
    names: List[BandName] = Field(description="Uma lista com os nomes das bandas.")

In [60]:
parser = PydanticOutputParser(pydantic_object=BandsNames)

In [71]:
def generate_band_name(genre: str, llm = llm) -> str:
    """ 
    """
    principles = """\
        - O nome precisa ser fácil de lembrar. 
        - O nome deve ser fácil de pronunciar. 
        - Usa o gênero {genero} e seu contexto cultural para criar um nome eficiente. 
        - Você deve retornar apenas o nome, sem nenhum outro texto qualquer. 
        - O comprimento máximo da sentença deve ser de 10 characteres.
        - Evite retornar full stops, como '\n' e outros characteres. 
    """

    template = """\
        Gere 5 nomes para bandas relacionadas ao gênero {genre}. 
        Você precisa seguir os seguintes princípios: {principles}. 

        # Instrunções #
        {format_instrunctions}
    """

    system_prompt = PromptTemplate(
        template=template, 
        input_variables=["format_instructions", "genre", "principles"]
    )

    chain = system_prompt | llm

    return chain.invoke(
        {
            "format_instrunctions": parser.get_format_instructions(),
            "genre": genre, 
            "principles": principles, 
        }
    ).content

In [ ]:
response = parser.parse(generate_band_name(genre="dark wave"))

In [77]:
pprint(response)

BandsNames(names=[BandName(name='MidnightVeil', rating_score=8.0), BandName(name='ShadowWeave', rating_score=9.0), BandName(name='DarkstarRise', rating_score=7.0), BandName(name='EchoFade', rating_score=8.0), BandName(name='VelvetRequiem', rating_score=9.0)])


### Extraindo features do texto

A extração de features do texto pode ser pensada da seguinte forma: existem certas características principais que formam um texto tal qual ele se apresenta, como tom de voz, comprimento do texto, estilo de escrita, tema principal e assim por diante. Conseguindo extrair as características principais de um texto o qual representa a ground truth a partir da qual o modelo pode passar a utilizar para gerar novos textos de forma acurada pode aumentar a qualidade desses. 

In [37]:
def analyse_text(query: str, llm = llm) -> str:
    """
    Função que utiliza de uma LLM para analisar as features do 
    texto e extrai-los, permitindo a criação de um novo.  

    Args:
        query (str): A mensagem de entrada
    Returns: 
        str: A resposta gerada pelo modelo.
    """
    template = """\
        Aja como um leitor experiente e analise minuciosamente o texto que lhe for passado. 
        A partir do texto, analise as características do texto, como tom de voz, estilo, 
        comprimento, tema e sentimento prevalente.

        Responda sempre em português. 

        Texto : {query}
        Análise: 
    """

    system_message = PromptTemplate(
        template        = template, 
        input_variables = ["query"]
    )

    chain = system_message | llm

    return chain.invoke(query).content

def create_based_text(query: str, llm = llm) -> str:
    """
    Função que utiliza de uma LLM para gerar um texto com base 
    na analise das características de um texto anterior e a sua 
    referência. 

    Args:
        query (str): A mensagem de entrada
    Returns: 
        str: A resposta gerada pelo modelo.
    """
    template = """\
        Aja como um escritor experiente e escreva um texto que lhe for pedido com base no texto informado, 
        utilizando-se das características identificadas, com o objetivo de criar um texto próprio.
        
        Responda sempre em português. 

        Texto : {query}
        Análise: 
    """

    system_message = PromptTemplate(
        template        = template, 
        input_variables = ["query"]
    )

    chain = system_message | llm

    return chain.invoke(query).content

In [23]:
text = """\
    Ho pensato tanto
    Forse pure troppo, dimmi perché
    Credevo fosse vero
    Sembravi così sincero ma
    Eri solo tu, nient'altro di più
    Ho imparato tanto
    E tutto troppo presto, ma adesso lo so
    È sembrato così strano
    Accettare quella che sono da
    Allontanarmi da me
    Quello che ancora non c'è
    Arriverà da sé
    Non aver paura che
    Non ci sia tempo per te
    Non cercare fuori
    Quello che è dentro di te
    E anche se non ti avrò mai
    E mi va bene uguale
    Anche se mi fa male
    Ora lasciami in pace
    Ho pensato tanto
    Così tanto che non mi è venuto in mente niente
    Forse perché
    Vorrei esperienze dense e non da collezione
    Senza trattenere sempre ogni mia emozione
    Distrarmi un secondo da me
    Quello che ancora non c'è
    Arriverà da sé
    Non aver paura che
    Non ci sia tempo per te
    Non cercare fuori
    Quello che è dentro di te
    E anche se non ti avrò mai
    Mi va bene uguale
    Anche se mi fa male
    Scusa se ti ho rotto
    Col mio sogno mezzo rotto
    Ma lo sai che sento troppo
    E si amplifica quello che sento
    Sembra solo un'altra onda che si infrange
    Che ci prova anche se alla fine non ha senso e piange
    Ma ci prova perché è nella sua natura
    E ora lascia che sia il tempo a cucire questa spaccatura
    Ripetiamo parole, legittimiamo distanze
    E le usiamo per colmare il vuoto nelle nostre stanze
    Quello che ancora non c'è
    Arriverà da sé
    Non aver paura che
    Non ci sia tempo per te
    Non cercare fuori
    Quello che è dentro di te
    E anche se non ti avrò mai
    Mi va bene uguale
    Anche se mi fa male
    Ora lasciami andare
"""

analyse = analyse_text(query=text)
Markdown(analyse)

Como leitor experiente, farei uma análise minuciosa do texto.

**Tom de voz:** O tom de voz do texto é introspectivo, reflexivo e emocional. O autor parece estar fazendo uma introspecção profunda sobre si mesmo e suas experiências, compartilhando seus pensamentos e sentimentos de forma honesta e aberta.

**Estilo:** O estilo do texto é lírico e poético, com uma linguagem rica e evocativa. O autor usa metáforas, imagens e expressões figuradas para transmitir seus sentimentos e ideias. O texto também apresenta uma estrutura não linear, com versos que se repetem e se entrelaçam, criando uma sensação de fluxo de consciência.

**Comprimento:** O texto é relativamente longo, com 32 versos, o que sugere que o autor está disposto a explorar seus pensamentos e sentimentos em profundidade.

**Tema:** O tema principal do texto é a introspecção e a auto-reflexão. O autor está procurando entender a si mesmo, suas experiências e seus sentimentos, e está compartilhando essa jornada com o leitor. Outros temas presentes no texto incluem a perda, a dor, a aceitação e a busca por autenticidade.

**Sentimento prevalente:** O sentimento prevalente do texto é de melancolia e introspecção. O autor parece estar lidando com sentimentos de perda e dor, mas também há uma sensação de aceitação e resignação. O texto também transmite uma sensação de esperança e otimismo, com o autor incentivando o leitor a buscar dentro de si mesmo e a não ter medo do futuro.

**Outras características:** O texto apresenta uma linguagem muito pessoal e íntima, com o autor usando pronome "eu" e "me" para se referir a si mesmo. Isso cria uma sensação de proximidade e intimidade com o leitor. Além disso, o texto apresenta uma estrutura musical, com versos que se repetem e se entrelaçam, criando uma sensação de ritmo e fluxo.

**Conclusão:** Em resumo, o texto é uma reflexão profunda e emocional sobre a vida, a perda e a busca por autenticidade. O autor apresenta uma linguagem lírica e poética, com uma estrutura não linear e uma linguagem muito pessoal. O sentimento prevalente do texto é de melancolia e introspecção, mas também há uma sensação de esperança e otimismo.

In [38]:
text_ = f"""\
    Escreva um poema inspirado em {text}. 
    Utiliza-se da análise presente em {analyse} para a sua construnção.
    Não copie o texto informado, mas apenas inspira-se nele, criando um novo
    poema no lugar. 
"""

create_based_text = create_based_text(text_)
Markdown(create_based_text)

**Poema Inspirado**

Eu me perco em meus pensamentos
Tanto que não sei mais onde estou
Procuro respostas em meus sonhos
Mas elas fogem como areia entre meus dedos

Eu creio que era verdade
Mas era apenas uma ilusão
Eu me perdi em meus desejos
E agora estou sozinho, sem rumo

Eu aprendi tanto
Mas não o suficiente
Eu sinto que estou longe de mim
E que preciso encontrar meu caminho

Eu me sinto estranho
Aceitando quem eu sou
Me afastando de mim mesmo
E encontrando o que ainda não é

Eu sei que não há tempo para mim
Mas não tenho medo
Eu sei que o que está dentro de mim
É o que verdadeiramente importa

Eu não procuro mais fora
Eu busco dentro de mim
Eu sei que o que eu sinto
É o que me faz ser quem eu sou

Eu sinto dor e tristeza
Mas também sinto esperança
Eu sei que um dia encontrarei
O que ainda não é

Eu peço desculpas se te machuquei
Com meus sonhos quebrados
Mas eu sinto demais
E isso me faz ser quem eu sou

Eu deixo que o tempo cuide
Das feridas que eu tenho
Eu sei que um dia estarei bem
E encontrarei meu caminho

Eu repito palavras
E legitimo distâncias
Eu uso elas para preencher
O vazio que eu sinto

Eu sei que um dia encontrarei
O que ainda não é
Eu não tenho medo
Eu sei que o que está dentro de mim
É o que verdadeiramente importa.

Este poema inspirado mantém o tom introspectivo e emocional do texto original, com uma linguagem lírica e poética. A estrutura do poema é não linear, com versos que se repetem e se entrelaçam, criando uma sensação de fluxo de consciência. O tema principal do poema é a introspecção e a auto-reflexão, com o autor procurando entender a si mesmo e suas experiências. O sentimento prevalente do poema é de melancolia e introspecção, mas também há uma sensação de esperança e otimismo.